# Milestone 1 — Data Exploration

**Dataset:** US Flight Delays 2015 (3 files in `data/`):
- `flights.csv` — main table (~5.8M flights)
- `airlines.csv` — airline code → name
- `airports.csv` — airport code → city, state, coordinates

**Goal:** understand the data and decide:
- how to define "delayed" (target);
- which columns can be used as features;
- which columns must not be used (data leakage).

In [0]:
import pandas as pd
from pathlib import Path

#main path
DATA_DIR = Path("../data")

In [0]:
# flights.csv is very large (~500 MB), so a lower sized sample created.
SAMPLE_ROWS = 200_000
#===========================================================
flights = pd.read_csv(DATA_DIR / "flights_sample.csv", nrows=SAMPLE_ROWS)
airlines = pd.read_csv(DATA_DIR / "airlines.csv")
airports = pd.read_csv(DATA_DIR / "airports.csv")

print(f"flights:  {flights.shape[0]:,} rows, {flights.shape[1]} columns (sample)")
print(f"airlines: {airlines.shape[0]} airlines")
print(f"airports: {airports.shape[0]} airports")

## 1. First look

In [0]:
flights.head()

In [0]:
print("Columns:")
print(flights.columns.tolist())

## 2. Missing values and types

In [0]:
missing = flights.isnull().sum().sort_values(ascending=False)
missing[missing > 0]

## 3. Delay-related columns

In [0]:
delay_cols = ["DEPARTURE_DELAY",
    "ARRIVAL_DELAY", "TAXI_OUT",
    "TAXI_IN", "AIR_SYSTEM_DELAY",
    "SECURITY_DELAY", "AIRLINE_DELAY",
    "LATE_AIRCRAFT_DELAY", "WEATHER_DELAY",]
#needed for data analysis
flights[delay_cols].describe()

## 4. Target — delayed or not

Rule: `is_delayed = 1` if delay is **≥ 15 minutes**, otherwise `0`.

You can use departure delay (`DEPARTURE_DELAY`) or arrival delay (`ARRIVAL_DELAY`).
We start with **departure delay** — easier to explain.

In [0]:
flights["is_delayed"] = (flights["DEPARTURE_DELAY"] >= 15).astype(int)
#at least 15 mins implies delayed
print(flights["is_delayed"].value_counts())
print()
print("Share of delayed flights:")
print(flights["is_delayed"].value_counts(normalize=True).round(3))

## 5. Cancelled and diverted flights

In [0]:
print("Cancelled (CANCELLED):", flights["CANCELLED"].value_counts().to_dict())
print("Diverted (DIVERTED):", flights["DIVERTED"].value_counts().to_dict())

## 6. Lookup tables: airlines and airports

In [0]:
airlines.head()

In [0]:
airports.head()

## 7. Which columns can / cannot be used

| Column | OK? | Why |
|--------|-----|-----|
| YEAR, MONTH, DAY, DAY_OF_WEEK | yes | known in advance |
| AIRLINE | yes | known before departure |
| FLIGHT_NUMBER | yes | known before departure |
| ORIGIN_AIRPORT, DESTINATION_AIRPORT | yes | route known in advance |
| SCHEDULED_DEPARTURE, SCHEDULED_ARRIVAL | yes | schedule known in advance |
| DISTANCE | yes | route distance known in advance |
| DEPARTURE_DELAY, ARRIVAL_DELAY | **no** | this is the delay (target or leakage) |
| DEPARTURE_TIME, ARRIVAL_TIME | **no** | actual times — only known after |
| TAXI_OUT, WHEELS_OFF, WHEELS_ON, TAXI_IN | **no** | happens during/after the flight |
| ELAPSED_TIME, AIR_TIME | **no** | only known after the flight |
| AIR_SYSTEM_DELAY, WEATHER_DELAY, ... | **no** | delay reasons — after the delay happened |
| CANCELLED, DIVERTED | **no** | flight outcome, not a pre-departure feature |
| TAIL_NUMBER | mb, not sure | usually skipped at the start |

## 8. Summary (Milestone 1)

Fill in after running the notebook with your numbers:

1. **Volume:** sample ~200k rows; full file ~5.8M flights for 2015.
2. **Target:** `is_delayed` from `DEPARTURE_DELAY >= 15`.
3. **Class balance:** check the percentage above — classes are likely imbalanced.
4. **Features for modeling (initial list):** AIRLINE, ORIGIN_AIRPORT, DESTINATION_AIRPORT, MONTH, DAY_OF_WEEK, DISTANCE, SCHEDULED_DEPARTURE (hour).
5. **Next step:** Milestone 2 — load `flights.csv` into Bronze Delta via Spark.